In [1]:
print("Ready to build an order book")
2 + 2

Ready to build an order book


4

## Prise de note / Document de Synthese



Question 1 : 
Ton order book a deux côtés, les bids (acheteurs) et les asks (vendeurs). Prenons juste le côté ask pour simplifier.

À un instant donné, ton côté ask contient plusieurs ordres, par exemple :

ordre A : vendre 5 unités à 101
ordre B : vendre 3 unités à 101
ordre C : vendre 2 unités à 102
ordre D : vendre 4 unités à 103

Ma question : quelles informations tu dois obligatoirement stocker pour chaque ordre ? Liste-moi tous les champs qui te viennent à l'esprit, sans en oublier.

Ma reponse : Pour chaque ordre, il faut stocker :
    - Le prix de l'ordre
    - Le nombre d'unites de l'ordre
    - Le temps auquel il a ete ajoute
    - A qui l'ordre appartient
    - Un order ID
    - order = new_order(price, time, unite amount)

# Document de Synthese

## Data structure d'un order

- order_id : identifiant unique de l'ordre
- trader_id : à qui il appartient
- side : buy ou sell (j'ajoute celui-là, tu l'avais implicitement mais il faut l'écrire noir sur blanc)
- price : le prix limite de l'ordre
- quantity : la quantité demandée
- sequence_number (ou timestamp) : l'ordre d'arrivée, pour la priorité temporelle
- Dans la quantity, il peut y avoir deux types : original_quantity et updated_quantity

Deux niveaux : dict de prix + deque par prix
- Niveau 1 (prix) : sert la priorité prix
- Niveau 2 (deque) : sert la priorité temps
- Ensemble : price-time priority par construction

## Algorithme de Matching 


**L'algorithme de matching, en français.**

Voici la logique que ta méthode va suivre pour un buy order qui arrive. Lis-la lentement :

1. Tant qu'il reste de la quantité à exécuter dans l'ordre entrant, ET qu'il y a des asks dans le carnet, ET que le meilleur ask (prix le plus bas) est inférieur ou égal au prix de l'ordre entrant (donc les prix se croisent) : on continue à matcher.

2. À chaque tour de boucle : on trouve le meilleur prix ask, on prend le premier ordre de la file à ce prix (celui arrivé en premier, priorité temporelle), on exécute un trade entre les deux.

3. Un trade s'exécute au prix de l'ordre déjà dans le carnet (le "resting order"), pas au prix de l'ordre entrant. Ça c'est une règle qu'on avait vue.

4. La quantité échangée est le minimum des deux quantités restantes. Si le buy veut 10 et le sell offre 5, on exécute 5.

5. On met à jour les deux ordres : on retire la quantité échangée. Si le resting order arrive à zéro, on l'enlève de la file. S'il reste de la quantité, il reste dans le carnet.

6. Une fois la boucle terminée, s'il reste de la quantité dans l'ordre entrant (personne ne veut plus lui vendre à son prix), il devient à son tour un resting order et va se poser dans les bids.

Pour un sell order, c'est symétrique : on cherche le meilleur bid (prix le plus haut), et on matche tant que le meilleur bid est supérieur ou égal au prix du sell entrant.


In [5]:
from collections import deque
from dataclasses import dataclass, field

In [6]:
@dataclass
class Order:
    order_id: int
    trader_id: int
    side: str  # buy ou sell
    price: float
    quantity: int
    sequence: int #numero d'arrivee
    

In [7]:
o = Order(order_id=1, trader_id="Alice", side="sell", price=101, quantity=5, sequence=1)
print(o)

Order(order_id=1, trader_id='Alice', side='sell', price=101, quantity=5, sequence=1)


In [13]:
class Order_book:
    def __init__(self):
        self.bids={} # dict{prix : deque d'ordres}, cote acheteurs
        self.asks={} # dict{prix : deque d'ordres}, cote vendeurs
        self.next_order_id=1
        self.next_sequence=1
        self.trades=[] # liste des trades executes, pour l'historique

    def _add_resting_order(self, order):
        """Ajoute l'ordre dans le carnet, à la fin de la file de son prix."""
        book_side = self.bids if order.side == "buy" else self.asks
        if order.price not in book_side:
            book_side[order.price] = deque()
        book_side[order.price].append(order)

    def submit(self, order):
        if order.side == "buy":
            book = self.asks
            while order.quantity != 0 and min(book) <= order.price and book :
                min_price = min(book)
                first_ask_order = book[min_price][0]
                quantity_traded = min(first_ask_order.quantity, order.quantity)
                order.quantity -= quantity_traded
                if first_ask_order.quantity -= quantity_traded == 0:
                    book[min_price].popleft()           
           if order.quantity != 0:
               _adding_resting_order(self, order)
        else:
            # sell à venir
            pass

In [20]:
book = Order_book()

# Créons quelques ordres manuellement
o1 = Order(order_id=1, trader_id="alice", side="sell", price=101, quantity=5, sequence=1)
o2 = Order(order_id=2, trader_id="bob",   side="sell", price=101, quantity=3, sequence=2)
o3 = Order(order_id=3, trader_id="carol", side="sell", price=102, quantity=2, sequence=3)

# On les ajoute dans le carnet
book._add_resting_order(o1)
book._add_resting_order(o2)
book._add_resting_order(o3)

# Regardons ce que ça donne
print("Asks :", book.asks)

Asks : {101: deque([Order(order_id=1, trader_id='alice', side='sell', price=101, quantity=5, sequence=1), Order(order_id=2, trader_id='bob', side='sell', price=101, quantity=3, sequence=2)]), 102: deque([Order(order_id=3, trader_id='carol', side='sell', price=102, quantity=2, sequence=3)])}
